# Train MLP Models for Federated Clients

Set the client data folder in the first code cell, then run the notebook. It trains one MLP per client using the same shared tuned/configurable architecture and training setup as `train_global_mlp.ipynb`.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

import torch

# Change this to the folder that contains client_000/client.csv, client_001/client.csv, ...
CLIENT_DATA_DIR = Path("../data/IID_cl3_age_unbalanced")

TARGET_COL = "default payment next month"
ACTIVE_FEATURE_SET = "engineered"
MODEL_BASE_DIR = Path("../models")

repo_root = Path.cwd()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not CLIENT_DATA_DIR.exists():
    candidate_client_dir = repo_root / "data" / CLIENT_DATA_DIR.name
    if candidate_client_dir.exists():
        CLIENT_DATA_DIR = candidate_client_dir
MODEL_BASE_DIR = repo_root / "models"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dataset.credit_data import prepare_feature_sets
from src.training.credit_mlp import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_MAX_EPOCHS,
    DEFAULT_PATIENCE,
    DEFAULT_SEED,
    make_loader,
    load_mlp_config,
    predict_proba,
    train_credit_default_mlp,
)

SEED = DEFAULT_SEED
BATCH_SIZE = DEFAULT_BATCH_SIZE
MAX_EPOCHS = DEFAULT_MAX_EPOCHS
PATIENCE = DEFAULT_PATIENCE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Client data folder:", CLIENT_DATA_DIR.resolve())


Device: cuda
Client data folder: /homes/sguszausky/fairfed-credit-xai/data/IID_cl3_age_unbalanced


In [2]:
# The fixed MLP architecture, loaders, and training loop live in credit_mlp.py.
# This notebook only discovers client files, trains one shared-base model per client, and saves artifacts.


In [3]:
def discover_client_files(client_data_dir):
    client_data_dir = Path(client_data_dir)
    nested_client_csvs = sorted(client_data_dir.glob("client_*/client.csv"))
    if nested_client_csvs:
        return nested_client_csvs

    flat_client_csvs = sorted(client_data_dir.glob("client_*.csv"))
    if flat_client_csvs:
        return flat_client_csvs

    raise FileNotFoundError(
        f"No client CSV files found under {client_data_dir}. Expected client_000/client.csv or client_000.csv."
    )


client_files = discover_client_files(CLIENT_DATA_DIR)
model_output_dir = MODEL_BASE_DIR / CLIENT_DATA_DIR.resolve().name
model_output_dir.mkdir(parents=True, exist_ok=True)
GLOBAL_MODEL_CONFIG_PATH = MODEL_BASE_DIR / "credit_default_mlp_config.json"
shared_model_config = load_mlp_config(GLOBAL_MODEL_CONFIG_PATH)

print(f"Found {len(client_files)} clients")
print("Model output folder:", model_output_dir.resolve())
print("Shared model config:", shared_model_config)
print("Config source:", GLOBAL_MODEL_CONFIG_PATH if GLOBAL_MODEL_CONFIG_PATH.exists() else "credit_mlp.py defaults")
for client_file in client_files:
    print("-", client_file)


Found 3 clients
Model output folder: /homes/sguszausky/fairfed-credit-xai/models/IID_cl3_age_unbalanced
Shared model config: {'hidden_dims': (256, 128, 64), 'dropout': (0.1, 0.1, 0.1), 'batch_norm': False, 'batch_size': 128, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'use_pos_weight': True, 'lr': 0.001}
Config source: /homes/sguszausky/fairfed-credit-xai/models/credit_default_mlp_config.json
- ../data/IID_cl3_age_unbalanced/client_000/client.csv
- ../data/IID_cl3_age_unbalanced/client_001/client.csv
- ../data/IID_cl3_age_unbalanced/client_002/client.csv


In [4]:
def train_one_client(client_csv, model_path):
    client_csv = Path(client_csv)
    model_path = Path(model_path)

    df = pd.read_csv(client_csv)
    feature_sets = prepare_feature_sets(df, target_col=TARGET_COL, seed=SEED)
    active_data = feature_sets[ACTIVE_FEATURE_SET]

    y_train = active_data["y_train"]
    y_val = active_data["y_val"]
    y_test = active_data["y_test"]
    test_loader = make_loader(active_data["X_test"], y_test, batch_size=shared_model_config["batch_size"], shuffle=False)

    training_result = train_credit_default_mlp(
        active_data,
        device=device,
        seed=SEED,
        model_config=shared_model_config,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        verbose=False,
    )

    model = training_result["model"]
    best_threshold = training_result["best_threshold"]

    y_test_true, y_test_prob = predict_proba(model, test_loader, device)
    y_test_pred = (y_test_prob >= best_threshold).astype(int)

    test_metrics = {
        "test_roc_auc": float(roc_auc_score(y_test_true, y_test_prob)),
        "test_pr_auc": float(average_precision_score(y_test_true, y_test_prob)),
        "test_accuracy": float(accuracy_score(y_test_true, y_test_pred)),
        "test_f1": float(f1_score(y_test_true, y_test_pred, zero_division=0)),
        "test_precision": float(precision_score(y_test_true, y_test_pred, zero_division=0)),
        "test_recall": float(recall_score(y_test_true, y_test_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_test_true, y_test_pred).tolist(),
        "classification_report": classification_report(
            y_test_true,
            y_test_pred,
            target_names=["no default", "default"],
            zero_division=0,
            output_dict=True,
        ),
    }

    artifact = {
        "model_state_dict": model.state_dict(),
        "input_dim": active_data["X_train"].shape[1],
        "best_threshold": best_threshold,
        "feature_set": ACTIVE_FEATURE_SET,
        "use_feature_engineering": active_data["use_feature_engineering"],
        "engineered_cols": active_data["engineered_cols"],
        "categorical_cols": active_data["categorical_cols"],
        "numeric_cols": active_data["numeric_cols"],
        "client_csv": str(client_csv),
        "client_data_dir": str(CLIENT_DATA_DIR),
        "model_config": training_result["model_config"],
        "training_config": {
            "seed": SEED,
            "batch_size": training_result["model_config"]["batch_size"],
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "learning_rate": training_result["model_config"]["learning_rate"],
            "weight_decay": training_result["model_config"]["weight_decay"],
            "pos_weight": training_result["pos_weight"],
        },
        "history": training_result["history"],
        "validation_metrics": {
            "best_val_auc": training_result["best_val_auc"],
            "best_val_f1": training_result["best_val_f1"],
        },
        "test_metrics": test_metrics,
    }

    torch.save(artifact, model_path)

    return {
        "client": client_csv.parent.name if client_csv.name == "client.csv" else client_csv.stem,
        "rows": len(df),
        "train_rows": len(y_train),
        "val_rows": len(y_val),
        "test_rows": len(y_test),
        "default_rate": float(df[TARGET_COL].mean()),
        "epochs": len(training_result["history"]),
        "best_threshold": best_threshold,
        "best_val_auc": training_result["best_val_auc"],
        "best_val_f1": training_result["best_val_f1"],
        **{key: value for key, value in test_metrics.items() if key.startswith("test_")},
        "model_path": str(model_path),
    }


In [5]:
results = []

for idx, client_csv in enumerate(client_files, start=1):
    client_name = client_csv.parent.name if client_csv.name == "client.csv" else client_csv.stem
    model_path = model_output_dir / f"{client_name}.pt"
    print(f"\n[{idx}/{len(client_files)}] Training {client_name}")
    result = train_one_client(client_csv, model_path)
    results.append(result)
    print(
        f"Saved {model_path} | "
        f"test_auc={result['test_roc_auc']:.4f} "
        f"test_f1={result['test_f1']:.4f} "
        f"threshold={result['best_threshold']:.3f}"
    )

results_df = pd.DataFrame(results)
summary_path = model_output_dir / "client_model_summary.csv"
results_df.to_csv(summary_path, index=False)

print("\nSaved summary to", summary_path)
results_df



[1/3] Training client_000
Saved /homes/sguszausky/fairfed-credit-xai/models/IID_cl3_age_unbalanced/client_000.pt | test_auc=0.7632 test_f1=0.5934 threshold=0.485

[2/3] Training client_001
Saved /homes/sguszausky/fairfed-credit-xai/models/IID_cl3_age_unbalanced/client_001.pt | test_auc=0.7938 test_f1=0.5481 threshold=0.615

[3/3] Training client_002
Saved /homes/sguszausky/fairfed-credit-xai/models/IID_cl3_age_unbalanced/client_002.pt | test_auc=0.7463 test_f1=0.5275 threshold=0.445

Saved summary to /homes/sguszausky/fairfed-credit-xai/models/IID_cl3_age_unbalanced/client_model_summary.csv


,client,rows,train_rows,val_rows,test_rows,default_rate,epochs,best_threshold,best_val_auc,best_val_f1,test_roc_auc,test_pr_auc,test_accuracy,test_f1,test_precision,test_recall,model_path
0,client_000,1000,700,150,150,0.251000,17,0.485,0.788299,0.615385,0.763215,0.500623,0.753333,0.593407,0.500000,0.729730,/homes/sguszausky/fairfed-credit-xai/models/II...
1,client_001,3000,2100,450,450,0.236333,17,0.615,0.791096,0.554545,0.793769,0.601784,0.791111,0.548077,0.558824,0.537736,/homes/sguszausky/fairfed-credit-xai/models/II...
2,client_002,1000,700,150,150,0.217000,12,0.445,0.879047,0.631579,0.746292,0.432482,0.713333,0.527473,0.406780,0.750000,/homes/sguszausky/fairfed-credit-xai/models/II...
